# 3-1. N-gram의 저주

> **이 노트북에서 답할 질문**
> 앞 단어를 더 많이 볼수록 언어 모델이 좋아질 텐데, 왜 통계적 모델은 그렇게 하지 못했을까?

## 출발점

언어 모델이 하는 일은 딱 하나입니다.

$$P(w_t \mid w_1, w_2, \dots, w_{t-1})$$

"지금까지 나온 단어들이 이랬을 때, 다음 단어가 무엇일 확률은?"

이걸 그대로 계산하려면 **모든 길이의 모든 문장**에 대한 통계가 필요합니다. 불가능하죠.
그래서 통계적 언어 모델은 과감하게 자릅니다. **"앞의 N-1개만 보자."**

$$P(w_t \mid w_1, \dots, w_{t-1}) \approx P(w_t \mid w_{t-N+1}, \dots, w_{t-1})$$

- N=2 (bigram): 바로 앞 1개만 봄
- N=3 (trigram): 앞 2개를 봄
- N=5: 앞 4개를 봄

직관적으로 N이 클수록 좋아야 합니다. 문맥을 더 많이 보니까요.
**정말 그런지 직접 세어 보겠습니다.**

In [2]:
import sys, pathlib

# SelfStudy 루트를 찾아 sys.path 에 추가 (common.py 를 쓰기 위해)
ROOT = pathlib.Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "common.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "common.py").exists(), "SelfStudy 폴더 안에서 실행하세요"
sys.path.insert(0, str(ROOT))

from common import load_sentences, set_seed
set_seed(1234)

In [3]:
from collections import Counter, defaultdict

sents = load_sentences()
print(f"문장 수: {len(sents)}")
for s in sents[:3]:
    print(" ", s)

# 아주 단순한 토크나이저: 공백으로 자릅니다.
# (한국어는 원래 형태소 분석이 필요하지만, 지금은 개념 확인이 목적입니다.)
def tokenize(sent):
    return sent.replace(".", " .").split()

print()
print(tokenize(sents[0]))

문장 수: 50
  언어 모델은 다음에 올 단어를 예측하는 기계이다.
  예측을 잘 하려면 문장의 앞부분을 기억해야 한다.
  기억해야 할 앞부분이 길어질수록 문제는 어려워진다.

['언어', '모델은', '다음에', '올', '단어를', '예측하는', '기계이다', '.']


## 1. N-gram 세어 보기

N-gram 모델의 학습은 **그냥 세는 것**입니다. 경사 하강법도, 파라미터도 없습니다.

$$P(w_t \mid \text{context}) = \frac{C(\text{context}, w_t)}{C(\text{context})}$$

분자는 "이 문맥 뒤에 이 단어가 나온 횟수", 분모는 "이 문맥이 나온 횟수"입니다.

In [4]:
BOS, EOS = "<s>", "</s>"

def make_ngrams(sents, n):
    """문장 리스트에서 (문맥, 다음단어) 쌍을 만들어 셉니다."""
    counts = defaultdict(Counter)   # 문맥 -> Counter(다음 단어)
    for s in sents:
        toks = [BOS] * (n - 1) + tokenize(s) + [EOS]
        for i in range(n - 1, len(toks)):
            ctx = tuple(toks[i - n + 1:i])
            counts[ctx][toks[i]] += 1
    return counts

for n in (2, 3, 4, 5):
    c = make_ngrams(sents, n)
    total_ngrams = sum(sum(v.values()) for v in c.values())
    print(f"N={n}:  서로 다른 문맥 {len(c):5d}개,  전체 N-gram {total_ngrams}개")

N=2:  서로 다른 문맥   286개,  전체 N-gram 485개
N=3:  서로 다른 문맥   404개,  전체 N-gram 485개
N=4:  서로 다른 문맥   432개,  전체 N-gram 485개
N=5:  서로 다른 문맥   432개,  전체 N-gram 485개


### 여기서 첫 번째 신호

N이 커질수록 **서로 다른 문맥의 수가 전체 N-gram 수에 바짝 다가갑니다.**

이게 무슨 뜻일까요? 문맥 하나당 관측치가 평균 1개에 가까워진다는 뜻입니다.
즉 **"이 문맥 다음엔 항상 이 단어"** 라고 외워 버리는 것과 같습니다.

숫자로 확인해 봅시다.

In [ ]:
print(f"{'N':>3} | {'문맥당 평균 관측수':>16} | {'단 1번만 본 문맥 비율':>20}")
print("-" * 50)
for n in (2, 3, 4, 5, 6):
    c = make_ngrams(sents, n)
    obs_per_ctx = [sum(v.values()) for v in c.values()]
    avg = sum(obs_per_ctx) / len(obs_per_ctx)
    singleton = sum(1 for o in obs_per_ctx if o == 1) / len(obs_per_ctx)
    print(f"{n:>3} | {avg:>16.2f} | {singleton:>19.1%}")

## 2. 진짜 문제: 처음 보는 문맥

외우는 것 자체보다 심각한 문제가 있습니다.
**학습 때 못 본 문맥이 나오면 모델이 아무 말도 못 합니다.**

$C(\text{context}) = 0$ 이면 분모가 0이라 확률을 정의할 수조차 없습니다.

말뭉치를 학습용/평가용으로 나눠서, 평가 문장의 문맥 중 몇 %가 "처음 보는 것"인지 세어 봅시다.

In [ ]:
import random

shuffled = sents[:]
random.Random(0).shuffle(shuffled)
split = int(len(shuffled) * 0.8)
train_sents, test_sents = shuffled[:split], shuffled[split:]
print(f"학습 {len(train_sents)}문장 / 평가 {len(test_sents)}문장\n")

print(f"{'N':>3} | {'처음 보는 문맥 비율':>18} | {'처음 보는 (문맥,단어) 비율':>26}")
print("-" * 58)
for n in (2, 3, 4, 5):
    train_counts = make_ngrams(train_sents, n)
    unseen_ctx = unseen_pair = total = 0
    for s in test_sents:
        toks = [BOS] * (n - 1) + tokenize(s) + [EOS]
        for i in range(n - 1, len(toks)):
            ctx, w = tuple(toks[i - n + 1:i]), toks[i]
            total += 1
            if ctx not in train_counts:
                unseen_ctx += 1
                unseen_pair += 1
            elif train_counts[ctx][w] == 0:
                unseen_pair += 1
    print(f"{n:>3} | {unseen_ctx/total:>17.1%} | {unseen_pair/total:>25.1%}")

### 이것이 "N-gram의 저주"입니다

N을 키우면:
- 문맥은 더 정확해지지만 → **거의 모든 문맥이 처음 보는 것**이 됩니다
- 확률이 0이 되거나 정의되지 않습니다
- 문맥 공간의 크기는 $V^{N-1}$ 로 **지수적으로** 커집니다

어휘가 1만 개일 때 필요한 문맥 개수를 보면 규모가 실감 납니다.

In [ ]:
V = 10_000
print(f"어휘 크기 V = {V:,}\n")
for n in range(2, 7):
    space = V ** (n - 1)
    print(f"N={n}: 가능한 문맥 {space:>28,} 개")

print()
print(f"참고 - 한국어 위키 전체 토큰 수는 대략 10억(1e9) 규모입니다.")
print(f"N=4 만 되어도 가능한 문맥({V**3:,})이 데이터 양을 아득히 넘어섭니다.")

## 3. 임시방편: 스무딩

0 확률을 막기 위해 통계적 모델은 **스무딩(smoothing)** 을 씁니다.
가장 단순한 것은 모든 조합에 1씩 더해 주는 Laplace 스무딩입니다.

$$P(w \mid c) = \frac{C(c, w) + 1}{C(c) + V}$$

관측 안 된 조합에도 아주 작은 확률이 돌아갑니다. 문제는 **그 확률이 전부 똑같다**는 것.

In [ ]:
n = 3
train_counts = make_ngrams(train_sents, n)
vocab = sorted({t for s in sents for t in tokenize(s)} | {BOS, EOS})
V_real = len(vocab)

def p_laplace(ctx, w, counts, V):
    c = counts.get(ctx, Counter())
    return (c[w] + 1) / (sum(c.values()) + V)

ctx = tuple(tokenize(train_sents[0])[:2])
print(f"문맥: {ctx}\n")
probs = sorted(((p_laplace(ctx, w, train_counts, V_real), w) for w in vocab), reverse=True)
for p, w in probs[:5]:
    print(f"  P({w:<12} | ctx) = {p:.5f}")
print("  ...")
print(f"  관측되지 않은 단어들은 전부 {1/(sum(train_counts.get(ctx, Counter()).values())+V_real):.5f} 로 동일")

### 스무딩의 한계가 바로 여기 있습니다

`언어 모델은` 다음에 올 단어로 **`기계이다`** 와 **`고양이다`** 는 완전히 다른 그럴듯함을 가져야 합니다.
그런데 둘 다 관측되지 않았다면, Laplace 스무딩은 **똑같은 확률**을 줍니다.

모델이 "기계와 고양이는 다른 종류의 단어"라는 걸 표현할 방법이 전혀 없는 것입니다.
단어들이 서로 **완전히 독립적인 기호**이기 때문입니다.

---

## 4. 탈출구

문제의 뿌리는 단어를 **원자 기호**로 다룬다는 데 있습니다.
`기계`와 `장치`가 비슷하다는 정보가 모델 어디에도 없습니다.

해결책: **단어를 벡터로 바꾸자.**

비슷한 단어가 벡터 공간에서 가까이 있다면,
`언어 모델은 기계이다`로 배운 지식이 `언어 모델은 장치이다`로 **자동으로 번져 나갑니다**.
관측한 적 없는 조합인데도 그럴듯한 확률이 나오는 것이죠.

이것이 뉴럴 언어 모델의 핵심 아이디어이고, 다음 노트북들에서 하나씩 만들어 봅니다.

---

## 연습

1. `_CORPUS`(common.py) 대신 훨씬 큰 한국어 텍스트 파일을 만들어 `load_sentences(path)` 로 넘겨 보세요.
   말뭉치가 커지면 "처음 보는 문맥 비율"이 얼마나 떨어지나요? N=5에서도 충분히 떨어지던가요?
2. 공백 토크나이저 대신 **글자 단위**로 잘라 보세요. 어휘 크기 V가 줄어들면 희소성 문제는 어떻게 달라지나요?
3. Laplace 스무딩에서 1 대신 0.01을 더하면(add-k) 어떤 차이가 생기는지 확인해 보세요.

## 다음

→ `0302_Perplexity.ipynb` : 만든 모델이 얼마나 좋은지 **숫자로 재는 법**